In [ ]:
!pip install -U transformers datasets accelerate scikit-learn huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/NLP_Final_Project/Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
from huggingface_hub import notebook_login
notebook_login() # User will input their HF token here

In [ ]:
from datasets import load_dataset, concatenate_datasets

# 1. Load the Source Languages
print("Loading Hausa and Amharic datasets...")
dataset_hau = load_dataset("afrihate/afrihate", "hau")
dataset_amh = load_dataset("afrihate/afrihate", "amh")

# 2. Concatenate the sets
train_combined = concatenate_datasets([dataset_hau['train'], dataset_amh['train']])
test_source = concatenate_datasets([dataset_hau['test'], dataset_amh['test']])

# 3. Create strict Train, Dev(Eval), and Test splits
# Splitting the combined train set: 85% Train, 15% Dev/Eval
train_eval_split = train_combined.train_test_split(test_size=0.15, seed=42)

train_data = train_eval_split['train']
eval_data = train_eval_split['test']
test_data = test_source # Held-out test set for the source languages

print(f"Train size: {len(train_data)} | Eval size: {len(eval_data)} | Test size: {len(test_data)}")

In [ ]:
from transformers import AutoTokenizer

model_name = "Davlan/afro-xlmr-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Standardizing the text column. Adjust 'text' to 'tweet' if the dataset schema differs.
    text_column = "text" if "text" in examples else "tweet"
    return tokenizer(examples[text_column], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_eval = eval_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Automatically detect number of unique labels
num_labels = len(set(train_data['label']))
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",        # Evaluate frequently
    eval_steps=200,               # Run evaluation every 200 steps
    save_strategy="steps",        # Save checkpoints frequently to survive disconnects
    save_steps=200,               
    save_total_limit=3,           # Keep only the 3 most recent checkpoints to save Drive space
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Adjust to 8 if Colab Pro GPU runs out of VRAM
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,  # Automatically load the best checkpoint at the end
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

# Start training. If a checkpoint exists, it will resume automatically.
trainer.train()

In [ ]:
# Evaluate on the completely held-out source test set
print("Evaluating on held-out test set...")
test_results = trainer.evaluate(tokenized_test)
print(test_results)

# Save the final, fully-trained model to Google Drive
FINAL_MODEL_PATH = '/content/drive/MyDrive/NLP_Final_Project/Final_Source_Model'
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Final model saved securely to {FINAL_MODEL_PATH}")